# 01 — Uuriv andmeanalüüs (EDA)

Vastab viiele uurivale küsimusele Eesti tehnoülevaatuse andmetest.

| # | Küsimus |
|---|----------|
| Q1 | Mis keretüüpidel on parim/halvim läbimise määr aasta lõikes? |
| Q2 | Kes on rangeim inspektor? |
| Q3 | Millised sõidukitüübid koonduvad millistesse punktidesse? |
| Q4 | Millistele autodele on sarnased rikkeprofiilid? |
| Q5 | Vanim auto, mis igal kuul läbis / kukkus läbi? |

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# ── Year selection ────────────────────────────────────────────────────────────
# Fewer years = faster queries. Add/remove years as needed.
ANALYSIS_YEARS = list(range(2015, 2025))   # 10 years: 2015–2024

YEAR_URLS = {
    2010: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/9b2d3dbe-e35c-4b5f-baed-6990baa408d0/download-s3',
    2011: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/5139abc3-6823-4121-8d2c-0c82928ac8ac/download-s3',
    2012: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/53f52202-e94e-4cb8-9149-4e311e6f2fdb/download-s3',
    2013: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/3f859ecc-7296-4b9c-ae9b-625b95c90ad9/download-s3',
    2014: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/10103f5a-0b6d-46fc-bd16-99a5c433625e/download-s3',
    2015: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/788a9115-a5da-4f47-bbcc-c1c3b644d3b3/download-s3',
    2016: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/f4f0b51e-8343-4832-9c07-4c04448b8f21/download-s3',
    2017: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/b5a08ea9-0fa8-4cb5-b0bc-7109571d8de4/download-s3',
    2018: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/2d018cd8-f3d7-4242-8514-99633120992a/download-s3',
    2019: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/7ba50767-9844-40bb-8561-af89b634e201/download-s3',
    2020: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/6371e8dc-9906-4555-af9f-f927f2ccf938/download-s3',
    2021: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/f3fe9ef1-897c-45b3-b2dd-908b810aae9c/download-s3',
    2022: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/ba317d52-71b7-473d-bc87-aec0cde38434/download-s3',
    2023: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/1943aed4-8e53-4e70-9946-7fc8ad1f7dfe/download-s3',
    2024: 'https://andmed.eesti.ee/api/datasets/ae47fec7-63d0-4b7a-969b-fbdfed21d52a/files/af5b081a-3db1-495d-90f3-c334a860938a/download-s3',
    2025: 'https://pilv.transpordiamet.ee/s/Iiee4OAYFq4lT1v/download?path=%2F&files=yv_2025.csv',
}

def build_urls(years):
    urls = [YEAR_URLS[y] for y in sorted(years) if y in YEAR_URLS]
    quoted = ', '.join(f"'{u}'" for u in urls)
    return f'[{quoted}]'

URLS = build_urls(ANALYSIS_YEARS)
CSV_OPTS = "delim=',', header=true, encoding='utf-8'"

# ── Defect name lookup ────────────────────────────────────────────────────────
RIKE_PATH = Path('../data/raw/rike.csv')
if RIKE_PATH.exists():
    _rike = pd.read_csv(RIKE_PATH)
    _nimetus = _rike[_rike['TYYP'] == 'NIMETUS'][['ID', 'NIMETUS']].dropna()
    rike_lookup = dict(zip(_nimetus['ID'].astype(str), _nimetus['NIMETUS']))
else:
    rike_lookup = {}
    print('⚠ rike.csv not found — defect names will not be shown')

print(f'✓ Analüüsitavad aastad: {min(ANALYSIS_YEARS)}–{max(ANALYSIS_YEARS)} ({len(ANALYSIS_YEARS)} aastat)')
print(f'✓ Rikkekoodid laetud: {len(rike_lookup)}')
print('Esimene päring võib võtta 1–2 minutit (DuckDB laeb andmeid S3-st).')

---
## Q1 — Keretüübi mõju läbivaatuse tulemusele

**Eesmärk:** Iga keretüübi ja aasta kohta arvuta läbimise määr KORRALINE ülevaatustel.

**Väljund:** Kuumakaart — read = keretüübid, veerud = aastad, lahtrid = läbimise % 

In [ ]:
q1 = duckdb.sql(f"""
    SELECT
        CAST(SUBSTR(YV_KUUPAEV, 1, 4) AS INTEGER)                   AS aasta,
        KERETYYP,
        COUNT(*)                                                     AS kokku,
        SUM(CASE WHEN YLEVAATUSOTSUS='KORRAS' THEN 1 ELSE 0 END)    AS labis,
        ROUND(100.0 *
            SUM(CASE WHEN YLEVAATUSOTSUS='KORRAS' THEN 1 ELSE 0 END)
            / COUNT(*), 1)                                           AS labimise_protsent
    FROM read_csv_auto({URLS}, {CSV_OPTS})
    WHERE YLEVAATUSLIIK  = 'KORRALINE'
      AND YLEVAATUSOTSUS IN ('KORRAS', 'KORDUVALE')
      AND KERETYYP IS NOT NULL AND KERETYYP != ''
    GROUP BY aasta, KERETYYP
    HAVING COUNT(*) >= 200
    ORDER BY aasta, KERETYYP
""").df()

print(f'Ridu: {len(q1):,}  |  Keretüüpe: {q1["KERETYYP"].nunique()}  |  Aastaid: {q1["aasta"].nunique()}')
q1.head()

In [ ]:
# Filter to body types with enough data across years (at least 5 years present)
year_coverage = q1.groupby('KERETYYP')['aasta'].nunique()
valid_types = year_coverage[year_coverage >= 5].index
q1_filtered = q1[q1['KERETYYP'].isin(valid_types)].copy()

# Pivot for heatmap
pivot = q1_filtered.pivot_table(
    index='KERETYYP', columns='aasta', values='labimise_protsent'
).round(1)

# Sort rows by average pass rate
pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

fig = px.imshow(
    pivot,
    text_auto=True,
    color_continuous_scale='RdYlGn',
    zmin=60, zmax=95,
    labels=dict(x='Aasta', y='Keretüüp', color='Läbimise %'),
    title='Q1 — Läbimise määr (%) keretüübi ja aasta lõikes (KORRALINE ülevaatused)',
    aspect='auto',
)
fig.update_layout(height=500, coloraxis_colorbar=dict(title='Läbimise %'))
fig.show()

In [ ]:
# Also: trend over time per body type (line chart)
fig2 = px.line(
    q1_filtered,
    x='aasta', y='labimise_protsent', color='KERETYYP',
    markers=True,
    labels=dict(aasta='Aasta', labimise_protsent='Läbimise %', KERETYYP='Keretüüp'),
    title='Q1 — Läbimise määra trend keretüübi lõikes',
)
fig2.update_layout(height=450, yaxis_range=[50, 100])
fig2.show()

### Q1 Tõlgendus

*Täitke pärast diagrammide vaatamist.* Näiteks:
- Millisel keretüübil on järjepidevalt kõrgeim/madalaim läbimise määr?
- Kas on näha trendi läbi aastate (üldine paranemine/halvenemine)?
- Kas mõni keretüüp on oluliselt muutunud?

---
## Q2 — Rangeim inspektor

**Eesmärk:** Leida, millisel töötajal (TOOTAJA kood) on kõrgeim mitteminemise määr.

**Märkus:** TOOTAJA on anonüümne töötaja kood, mitte nimi.

**Väljund:** Top 20 rangeima ja leebema inspektori tulpdiagramm + koondtabel.

In [ ]:
q2 = duckdb.sql(f"""
    SELECT
        TOOTAJA                                                         AS inspektori_kood,
        TEHNOYLEVAATUSPUNKT                                             AS jaam,
        PUNKTI_KOOD                                                     AS jaama_kood,
        COUNT(*)                                                        AS kokku,
        SUM(CASE WHEN YLEVAATUSOTSUS='KORRAS'    THEN 1 ELSE 0 END)    AS labis,
        SUM(CASE WHEN YLEVAATUSOTSUS='KORDUVALE' THEN 1 ELSE 0 END)    AS kukkus,
        ROUND(100.0 *
            SUM(CASE WHEN YLEVAATUSOTSUS='KORRAS' THEN 1 ELSE 0 END)
            / COUNT(*), 1)                                              AS labimise_protsent,
        ROUND(100.0 *
            SUM(CASE WHEN YLEVAATUSOTSUS='KORDUVALE' THEN 1 ELSE 0 END)
            / COUNT(*), 1)                                              AS kukkumise_protsent
    FROM read_csv_auto({URLS}, {CSV_OPTS})
    WHERE YLEVAATUSLIIK  = 'KORRALINE'
      AND YLEVAATUSOTSUS IN ('KORRAS', 'KORDUVALE')
      AND TOOTAJA IS NOT NULL AND TOOTAJA != ''
    GROUP BY TOOTAJA, TEHNOYLEVAATUSPUNKT, PUNKTI_KOOD
    HAVING COUNT(*) >= 200
    ORDER BY kukkumise_protsent DESC
""").df()

print(f'Inspektoreid (min 200 ülevaatust): {len(q2):,}')
q2.head(10)

In [ ]:
top20_range = q2.head(20).copy()
top20_leebe = q2.tail(20).sort_values('kukkumise_protsent').copy()

fig, (ax1, ax2) = None, None  # using plotly subplots

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Top 20 rangeimat inspektorit (kõrgeim mitteminemise %)',
        'Top 20 leebeimat inspektorit (madalaim mitteminemise %)'
    )
)

fig.add_trace(
    go.Bar(
        x=top20_range['inspektori_kood'],
        y=top20_range['kukkumise_protsent'],
        name='Rangeim',
        marker_color='crimson',
        text=top20_range['jaama_kood'],
        textposition='outside',
        hovertemplate=(
            'Inspektor: %{x}<br>'
            'Mitteminemise %: %{y}<br>'
            'Jaam: %{text}<br>'
            '<extra></extra>'
        )
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=top20_leebe['inspektori_kood'],
        y=top20_leebe['kukkumise_protsent'],
        name='Leebeim',
        marker_color='steelblue',
        text=top20_leebe['jaama_kood'],
        textposition='outside',
        hovertemplate=(
            'Inspektor: %{x}<br>'
            'Mitteminemise %: %{y}<br>'
            'Jaam: %{text}<br>'
            '<extra></extra>'
        )
    ),
    row=1, col=2
)

fig.update_layout(
    title='Q2 — Inspektorite rangus (KORRALINE ülevaatused, min 200 ülevaatust)',
    height=500, showlegend=False
)
fig.update_xaxes(tickangle=45)
fig.update_yaxes(title_text='Mitteminemise %', row=1, col=1)
fig.update_yaxes(title_text='Mitteminemise %', row=1, col=2)
fig.show()

In [ ]:
# Summary table
display_cols = ['inspektori_kood', 'jaam', 'jaama_kood', 'kokku', 'labis', 'kukkus', 'labimise_protsent', 'kukkumise_protsent']
print('=== Top 20 rangeimat ===')
display(q2[display_cols].head(20).reset_index(drop=True))
print('\n=== Top 20 leebeimat ===')
display(q2[display_cols].tail(20).sort_values('kukkumise_protsent').reset_index(drop=True))

### Q2 Tõlgendus

*Täitke pärast andmete vaatamist.* Näiteks:
- Kui suur on vahe rangeima ja leebema inspektori vahel?
- Kas rangeimad inspektorid on koondunud kindlatesse punktidesse?
- Kas on näha geograafilist mustrit (jaama_kood)?

---
## Q3 — Millised sõidukitüübid koonduvad millistesse punktidesse?

**Eesmärk:** Iga punkti kohta leida domineeriv keretüüp — kas mõned punktid spetsialiseeruvad?

**Väljund:** Kuumakaart — punktid (read) × keretüübid (veerud), lahtrid = % punkti liiklusest.

In [ ]:
q3_raw = duckdb.sql(f"""
    SELECT
        PUNKTI_KOOD,
        TEHNOYLEVAATUSPUNKT                                             AS jaam,
        KERETYYP,
        COUNT(*)                                                        AS arv
    FROM read_csv_auto({URLS}, {CSV_OPTS})
    WHERE KERETYYP IS NOT NULL AND KERETYYP != ''
      AND PUNKTI_KOOD IS NOT NULL AND PUNKTI_KOOD != ''
    GROUP BY PUNKTI_KOOD, TEHNOYLEVAATUSPUNKT, KERETYYP
    ORDER BY PUNKTI_KOOD, arv DESC
""").df()

# Compute percentage per station
station_totals = q3_raw.groupby('PUNKTI_KOOD')['arv'].transform('sum')
q3_raw['pct'] = (q3_raw['arv'] / station_totals * 100).round(1)

# Keep top 20 stations by total volume
top_stations = (
    q3_raw.groupby('PUNKTI_KOOD')['arv'].sum()
    .nlargest(20).index
)
q3 = q3_raw[q3_raw['PUNKTI_KOOD'].isin(top_stations)].copy()

# Keep body types that appear in most stations
type_coverage = q3.groupby('KERETYYP')['PUNKTI_KOOD'].nunique()
top_types = type_coverage[type_coverage >= 5].index
q3 = q3[q3['KERETYYP'].isin(top_types)]

print(f'Punktid: {q3["PUNKTI_KOOD"].nunique()}  |  Keretüübid: {q3["KERETYYP"].nunique()}')

In [ ]:
# Add label: PUNKTI_KOOD + short station name
jaama_nimi = (
    q3_raw[q3_raw['PUNKTI_KOOD'].isin(top_stations)]
    .drop_duplicates('PUNKTI_KOOD')
    .set_index('PUNKTI_KOOD')['jaam']
)
q3['punkt_label'] = q3['PUNKTI_KOOD'].map(
    lambda k: f"{k} — {jaama_nimi.get(k, '')[:35]}"
)

pivot3 = q3.pivot_table(
    index='punkt_label', columns='KERETYYP', values='pct', fill_value=0
)

# Sort rows by dominant body type share (SEDAAN tends to be highest)
dominant = pivot3.idxmax(axis=1)
pivot3 = pivot3.sort_values('SEDAAN', ascending=False) if 'SEDAAN' in pivot3.columns else pivot3

fig3 = px.imshow(
    pivot3,
    text_auto='.1f',
    color_continuous_scale='Blues',
    labels=dict(x='Keretüüp', y='Tehnoülevaatuspunkt', color='% liiklusest'),
    title='Q3 — Keretüüpide jaotus punktide lõikes (% punkti kõigist ülevaatustest)',
    aspect='auto',
)
fig3.update_layout(height=600)
fig3.show()

In [ ]:
# Summary table: dominant type per station
dominant_table = (
    q3_raw[q3_raw['PUNKTI_KOOD'].isin(top_stations)]
    .sort_values('pct', ascending=False)
    .drop_duplicates('PUNKTI_KOOD')[['PUNKTI_KOOD', 'jaam', 'KERETYYP', 'arv', 'pct']]
    .rename(columns={'KERETYYP': 'domineeriv_keretyyp', 'pct': 'osakaal_%'})
    .sort_values('osakaal_%', ascending=False)
    .reset_index(drop=True)
)
print('Domineeriv keretüüp punktide lõikes (Top 20 punkti mahult):')
display(dominant_table)

### Q3 Tõlgendus

*Täitke pärast andmete vaatamist.* Näiteks:
- Kas mõni punkt on selgelt spetsialiseerunud (nt ainult kaubikud)?
- Kas enamik punktidest teenindab peamiselt sedaane ja universaale?
- Mis eristab kõige eripärasema profiiliga punkte?

---
## Q4 — Sarnase rikkeprofiilid sõidukite lõikes

**Eesmärk:** Leida 10 sagedasimat rikkekoodi ja vaadata, millised automargid jagavad sarnaseid rikkemustreid.

**Väljund:** Top-10 rikkete tabel + kuumakaart (mark × rike).

In [ ]:
# Step 1: Find top 10 defect IDs
top10_raw = duckdb.sql(f"""
    WITH exploded AS (
        SELECT
            TRIM(SPLIT_PART(TRIM(entry), ':', 1))   AS tase,
            TRIM(SPLIT_PART(TRIM(entry), ':', 2))   AS rike_id
        FROM (
            SELECT UNNEST(STRING_SPLIT(RIKKED, ',')) AS entry
            FROM read_csv_auto({URLS}, {CSV_OPTS})
            WHERE RIKKED IS NOT NULL AND RIKKED != ''
              AND YLEVAATUSLIIK = 'KORRALINE'
        )
        WHERE TRIM(entry) != ''
    )
    SELECT
        rike_id,
        tase    AS raskusaste,
        COUNT(*) AS esinemisi
    FROM exploded
    WHERE tase IN ('VO', 'OV', 'EOV')
      AND TRY_CAST(rike_id AS INTEGER) IS NOT NULL
    GROUP BY rike_id, tase
    ORDER BY esinemisi DESC
    LIMIT 10
""").df()

# Add defect names from lookup
top10_raw['nimetus'] = top10_raw['rike_id'].map(rike_lookup).fillna('(tundmatu)')
top10_ids = top10_raw['rike_id'].tolist()

print('Top 10 sagedasimat rikkekoodi:')
display(top10_raw[['rike_id', 'raskusaste', 'nimetus', 'esinemisi']])

In [ ]:
# Step 2: Build defect frequency matrix by MARK (per 1000 inspections)
id_list = ', '.join(f"'{d}'" for d in top10_ids)

q4_matrix = duckdb.sql(f"""
    WITH inspections AS (
        SELECT
            MARK,
            TRIM(SPLIT_PART(TRIM(entry), ':', 1))   AS tase,
            TRIM(SPLIT_PART(TRIM(entry), ':', 2))   AS rike_id
        FROM (
            SELECT MARK, UNNEST(STRING_SPLIT(RIKKED, ',')) AS entry
            FROM read_csv_auto({URLS}, {CSV_OPTS})
            WHERE RIKKED IS NOT NULL AND RIKKED != ''
              AND YLEVAATUSLIIK = 'KORRALINE'
              AND MARK IS NOT NULL AND MARK != ''
        )
        WHERE TRIM(entry) != ''
    ),
    mark_totals AS (
        SELECT MARK, COUNT(*) AS total_yv
        FROM read_csv_auto({URLS}, {CSV_OPTS})
        WHERE YLEVAATUSLIIK = 'KORRALINE'
          AND MARK IS NOT NULL AND MARK != ''
        GROUP BY MARK
        HAVING COUNT(*) >= 500
    )
    SELECT
        i.MARK,
        i.rike_id,
        COUNT(*)                                 AS rike_arv,
        m.total_yv,
        ROUND(1000.0 * COUNT(*) / m.total_yv, 2) AS per_1000
    FROM inspections i
    JOIN mark_totals m ON i.MARK = m.MARK
    WHERE i.tase IN ('VO', 'OV', 'EOV')
      AND i.rike_id IN ({id_list})
    GROUP BY i.MARK, i.rike_id, m.total_yv
    ORDER BY i.MARK, per_1000 DESC
""").df()

print(f'Marke (min 500 ülevaatust): {q4_matrix["MARK"].nunique()}')
q4_matrix.head()

In [ ]:
# Build heatmap: marks (rows) × top-10 defects (columns)
id_to_label = {
    row['rike_id']: f"{row['rike_id']}: {row['nimetus'][:30]}"
    for _, row in top10_raw.iterrows()
}
q4_matrix['rike_label'] = q4_matrix['rike_id'].map(id_to_label)

pivot4 = q4_matrix.pivot_table(
    index='MARK', columns='rike_label', values='per_1000', fill_value=0
)

# Sort marks by total defect frequency
pivot4 = pivot4.loc[pivot4.sum(axis=1).sort_values(ascending=False).head(30).index]

fig4 = px.imshow(
    pivot4,
    text_auto='.1f',
    color_continuous_scale='YlOrRd',
    labels=dict(x='Rikkekood', y='Automärk', color='Esinemisi / 1000 YV'),
    title='Q4 — Rikkeprofiilid märgi lõikes (esinemisi 1000 ülevaatuse kohta, Top 30 märki)',
    aspect='auto',
)
fig4.update_layout(height=700, xaxis_tickangle=30)
fig4.show()

In [ ]:
# Cosine similarity between marks based on defect profiles
from sklearn.metrics.pairwise import cosine_similarity
import plotly.figure_factory as ff

sim_matrix = cosine_similarity(pivot4.values)
sim_df = pd.DataFrame(sim_matrix, index=pivot4.index, columns=pivot4.index)

fig4b = px.imshow(
    sim_df,
    color_continuous_scale='Blues',
    zmin=0, zmax=1,
    labels=dict(x='Automärk', y='Automärk', color='Sarnasus'),
    title='Q4 — Rikkeprofiilide sarnasus automärkide vahel (kosiinussarnasus)',
)
fig4b.update_layout(height=650, width=700)
fig4b.show()

# Most similar pairs
pairs = []
for i, m1 in enumerate(sim_df.index):
    for j, m2 in enumerate(sim_df.columns):
        if i < j:
            pairs.append((m1, m2, sim_df.loc[m1, m2]))
pairs_df = pd.DataFrame(pairs, columns=['Mark 1', 'Mark 2', 'Sarnasus']).sort_values('Sarnasus', ascending=False)
print('Top 10 sarnaseima rikkeprofiilga märgipaar:')
display(pairs_df.head(10).reset_index(drop=True))

### Q4 Tõlgendus

*Täitke pärast andmete vaatamist.* Näiteks:
- Millised on kõige sagedasemad rikked üldiselt?
- Millistel märkidel on kõige sarnasemad rikkeprofiilid?
- Kas luksusautod ja eelarvesõidukid erinevad rikkeprofiililt?

---
## Q5 — Vanim auto, mis igal kuul läbis / kukkus läbi

**Eesmärk:** Iga aasta-kuu kohta leida vanim sõiduk, mis läbis KORRALINE ülevaatuse, ja vanim, mis kukkus läbi.

**Väljund:** Tabel + joondiagramm vanuse kohta läbi aastate.

In [ ]:
q5 = duckdb.sql(f"""
    WITH aged AS (
        SELECT
            YV_KUUPAEV,
            MARK, MUDEL, KERETYYP,
            TRY_CAST(ESMANE_REG_AASTA AS INTEGER)                    AS reg_aasta,
            CAST(SUBSTR(YV_KUUPAEV, 1, 4) AS INTEGER)
                - TRY_CAST(ESMANE_REG_AASTA AS INTEGER)              AS vanus,
            YLEVAATUSOTSUS,
            ROW_NUMBER() OVER (
                PARTITION BY YV_KUUPAEV, YLEVAATUSOTSUS
                ORDER BY TRY_CAST(ESMANE_REG_AASTA AS INTEGER) ASC
            ) AS rnk
        FROM read_csv_auto({URLS}, {CSV_OPTS})
        WHERE YLEVAATUSLIIK  = 'KORRALINE'
          AND YLEVAATUSOTSUS IN ('KORRAS', 'KORDUVALE')
          AND ESMANE_REG_AASTA IS NOT NULL
          AND TRY_CAST(ESMANE_REG_AASTA AS INTEGER) BETWEEN 1900 AND 2025
    )
    SELECT YV_KUUPAEV, MARK, MUDEL, KERETYYP, reg_aasta, vanus, YLEVAATUSOTSUS
    FROM aged
    WHERE rnk = 1
    ORDER BY YV_KUUPAEV, YLEVAATUSOTSUS
""").df()

# Split into pass / fail
q5_pass = q5[q5['YLEVAATUSOTSUS'] == 'KORRAS'].add_suffix('_labis').rename(columns={'YV_KUUPAEV_labis': 'kuu'})
q5_fail = q5[q5['YLEVAATUSOTSUS'] == 'KORDUVALE'].add_suffix('_kukkus').rename(columns={'YV_KUUPAEV_kukkus': 'kuu'})
q5_merged = q5_pass.merge(q5_fail, on='kuu', how='outer').sort_values('kuu')

print(f'Kuusid andmetes: {q5_merged["kuu"].nunique()}')
q5_merged[['kuu', 'MARK_labis', 'MUDEL_labis', 'reg_aasta_labis', 'vanus_labis',
           'MARK_kukkus', 'MUDEL_kukkus', 'reg_aasta_kukkus', 'vanus_kukkus']].head(12)

In [ ]:
# Line chart: oldest passing vs failing vehicle age over time
fig5 = go.Figure()
fig5.add_trace(go.Scatter(
    x=q5_merged['kuu'], y=q5_merged['vanus_labis'],
    mode='lines', name='Vanim läbinud', line=dict(color='green'),
    hovertemplate='%{x}<br>Vanus: %{y} a<br><extra>Läbis</extra>'
))
fig5.add_trace(go.Scatter(
    x=q5_merged['kuu'], y=q5_merged['vanus_kukkus'],
    mode='lines', name='Vanim kukkunud', line=dict(color='crimson'),
    hovertemplate='%{x}<br>Vanus: %{y} a<br><extra>Kukkus</extra>'
))
fig5.update_layout(
    title='Q5 — Vanima sõiduki vanus iga kuu KORRALINE ülevaatustel',
    xaxis_title='Kuu',
    yaxis_title='Sõiduki vanus (aastat)',
    height=450
)
fig5.show()

In [ ]:
# Most extreme cases
print('Top 5 vanimad sõidukid, mis LÄBISID KORRALINE ülevaatuse:')
display(
    q5_pass.nlargest(5, 'vanus_labis')
    [['kuu', 'MARK_labis', 'MUDEL_labis', 'reg_aasta_labis', 'vanus_labis', 'KERETYYP_labis']]
    .reset_index(drop=True)
)

print('\nTop 5 vanimad sõidukid, mis KUKKUSID LÄBI KORRALINE ülevaatuse:')
display(
    q5_fail.nlargest(5, 'vanus_kukkus')
    [['kuu', 'MARK_kukkus', 'MUDEL_kukkus', 'reg_aasta_kukkus', 'vanus_kukkus', 'KERETYYP_kukkus']]
    .reset_index(drop=True)
)

### Q5 Tõlgendus

*Täitke pärast andmete vaatamist.* Näiteks:
- Kui vana on vanim auto, mis regulaarsel ülevaatuse läbis?
- Kas väga vanad autod läbivad ülevaatuse tõenäolisemalt (omanikud on hoolikamad) või harvemini?
- Kas on näha sesoonsust (suvekuudel vanemad autod)?